In [ ]:
# ======================================================================
# Taller de Bases de Datos — Prototipo ejecutable (Google Colab)
# **Universidad ECCI · Ingeniería de Requerimientos**
#
# Equipo: Cristian Andrés Romero Gómez · Wilmer Alejandro Londoño Martínez · Axel
#
# Motor: **SQLite** (vía el módulo `sqlite3` de Python, no requiere instalación adicional en Colab)
#
# Este notebook crea, puebla y prueba la base de datos de:
#
# - **Caso 5 — Monitoreo de sensores IoT en una planta industrial**
#
# siguiendo el DDL, las reglas de negocio, las vistas, las consultas (4.7) y las pruebas de integridad/concurrencia (4.8) definidas en el expediente del taller.
#
# > No se usa ningún dato personal real: todos los nombres y valores de prueba son inventados, tal como exige la política de IA del taller.
# ======================================================================

# Celda 0 — Librerías necesarias
# Todas vienen preinstaladas en Google Colab, no hace falta pip install.
import sqlite3
import pandas as pd
import random
from datetime import datetime, timedelta

# `display` existe en Colab/Jupyter; si se corre como script normal (VS Code, terminal) se emula con print.
try:
    display
except NameError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("Librerías cargadas: sqlite3", sqlite3.sqlite_version, "| pandas", pd.__version__)

# ======================================================================
# # CASO 5 · Monitoreo de sensores IoT en una planta industrial
#
# Se crea la base `caso5_iot.db` con `maquina`, `sensor`, `lectura` (serie de tiempo),
# `lectura_resumen_hora` (agregación desnormalizada) y `alerta`, más las 2 vistas exigidas.
# Como el volumen real es ~43 millones de lecturas/día, aquí se simula una muestra representativa
# suficiente para probar las reglas de negocio y el rendimiento de los índices.
# ======================================================================

# 5.1 Conexión y DDL — Caso 5 (idéntico al script de la sección 4.6 del expediente)
conn5 = sqlite3.connect('caso5_iot.db')
conn5.execute("PRAGMA foreign_keys = ON;")
cur5 = conn5.cursor()

cur5.executescript('''
DROP VIEW IF EXISTS vista_sensores_sin_reportar;
DROP VIEW IF EXISTS vista_alertas_pendientes;
DROP TABLE IF EXISTS alerta;
DROP TABLE IF EXISTS lectura_resumen_hora;
DROP TABLE IF EXISTS lectura;
DROP TABLE IF EXISTS sensor;
DROP TABLE IF EXISTS maquina;

CREATE TABLE maquina (
  id_maquina INTEGER PRIMARY KEY,
  nombre TEXT NOT NULL,
  area TEXT NOT NULL
);

CREATE TABLE sensor (
  id_sensor INTEGER PRIMARY KEY,
  id_maquina INTEGER NOT NULL REFERENCES maquina(id_maquina),
  tipo TEXT NOT NULL,
  limite_min REAL NOT NULL,
  limite_max REAL NOT NULL
);

CREATE TABLE lectura (
  id_sensor INTEGER NOT NULL REFERENCES sensor(id_sensor),
  momento DATETIME NOT NULL,
  valor REAL NOT NULL,
  PRIMARY KEY (id_sensor, momento)
);
CREATE INDEX idx_lectura_momento ON lectura (momento);

CREATE TABLE lectura_resumen_hora (
  id_sensor INTEGER NOT NULL REFERENCES sensor(id_sensor),
  hora DATETIME NOT NULL,
  promedio REAL, minimo REAL, maximo REAL,
  PRIMARY KEY (id_sensor, hora)
);

CREATE TABLE alerta (
  id_alerta INTEGER PRIMARY KEY,
  id_sensor INTEGER NOT NULL REFERENCES sensor(id_sensor),
  momento DATETIME NOT NULL,
  valor REAL NOT NULL,
  estado TEXT NOT NULL CHECK (estado IN ('pendiente','atendida'))
);

CREATE VIEW vista_alertas_pendientes AS
  SELECT a.id_alerta, a.id_sensor, s.id_maquina, a.momento, a.valor
  FROM alerta a JOIN sensor s ON a.id_sensor = s.id_sensor
  WHERE a.estado = 'pendiente';

CREATE VIEW vista_sensores_sin_reportar AS
  SELECT s.id_sensor, MAX(l.momento) AS ultima_lectura
  FROM sensor s LEFT JOIN lectura l ON s.id_sensor = l.id_sensor
  GROUP BY s.id_sensor;
''')
conn5.commit()
print("Caso 5: tablas, vistas e índices creados en caso5_iot.db")

# 5.2 Datos de prueba: máquinas y sensores (inventados)
maquinas = [
    (1, "Extrusora A", "Producción"),
    (2, "Compresor B", "Utilities"),
]
sensores = [
    (1, 1, "temperatura", 20.0, 90.0),
    (2, 1, "vibracion", 0.0, 5.0),
    (3, 2, "presion", 1.0, 10.0),
]
cur5.executemany("INSERT INTO maquina VALUES (?,?,?)", maquinas)
cur5.executemany("INSERT INTO sensor VALUES (?,?,?,?,?)", sensores)
conn5.commit()
print(f"Insertadas {len(maquinas)} máquinas y {len(sensores)} sensores")

# 5.3 Simulación de lecturas de alto volumen (submuestra representativa)
# En producción real serían ~43 millones de lecturas/día; aquí se generan varias
# horas de lecturas cada segundo por sensor, suficiente para probar índices y agregación.
random.seed(42)
inicio = datetime(2026, 9, 10, 8, 0, 0)
lecturas = []
alertas_generadas = []
id_alerta = 1

rangos = {1: (20.0, 90.0), 2: (0.0, 5.0), 3: (1.0, 10.0)}
centro = {1: 55, 2: 2.5, 3: 5.5}

SEGUNDOS_SIMULADOS = 900  # 15 minutos de datos por sensor -> suficiente para probar sin saturar Colab

for s in (1, 2, 3):
    lim_min, lim_max = rangos[s]
    for seg in range(SEGUNDOS_SIMULADOS):
        momento = inicio + timedelta(seconds=seg)
        # ruido normal alrededor del centro, con picos ocasionales fuera de rango
        valor = round(random.gauss(centro[s], (lim_max - lim_min) * 0.1), 2)
        if random.random() < 0.002:  # 0.2% de lecturas simulan una falla real
            valor = lim_max + random.uniform(1, 5)
        lecturas.append((s, momento.strftime("%Y-%m-%d %H:%M:%S"), valor))
        if valor < lim_min or valor > lim_max:
            alertas_generadas.append((id_alerta, s, momento.strftime("%Y-%m-%d %H:%M:%S"), valor, "pendiente"))
            id_alerta += 1

cur5.executemany("INSERT INTO lectura VALUES (?,?,?)", lecturas)
cur5.executemany("INSERT INTO alerta VALUES (?,?,?,?,?)", alertas_generadas)
conn5.commit()
print(f"Insertadas {len(lecturas)} lecturas y {len(alertas_generadas)} alertas (RF-02 / RN-01)")

# 5.4 Prueba de integridad I-02 (RN-02): no duplicar lectura del mismo sensor
# con la misma marca de tiempo -> debe fallar por PRIMARY KEY (id_sensor, momento)
momento_dup = lecturas[0][1]
try:
    cur5.execute("INSERT INTO lectura VALUES (?,?,?)", (1, momento_dup, 99.9))
    conn5.commit()
    print("ERROR: se permitió una lectura duplicada (no debería pasar)")
except sqlite3.IntegrityError as e:
    print("OK — la base rechazó la lectura duplicada como se esperaba:")
    print(" ", e)

# 5.5 Prueba I-03 (RN-03): una alerta atendida no puede volver a marcarse como pendiente
def atender_alerta(cur, id_alerta):
    cur.execute("SELECT estado FROM alerta WHERE id_alerta = ?", (id_alerta,))
    row = cur.fetchone()
    if row is None:
        raise ValueError("Alerta no existe")
    if row[0] == "atendida":
        raise ValueError("La alerta ya fue atendida, no puede reabrirse (RN-03)")
    cur.execute("UPDATE alerta SET estado = 'atendida' WHERE id_alerta = ?", (id_alerta,))

if alertas_generadas:
    primer_id = alertas_generadas[0][0]
    atender_alerta(cur5, primer_id)
    conn5.commit()
    print(f"Alerta {primer_id} marcada como atendida")
    try:
        atender_alerta(cur5, primer_id)
        print("ERROR: se permitió reabrir una alerta atendida")
    except ValueError as e:
        print("OK —", e)
else:
    print("No se generaron alertas en esta simulación, ajustar semilla o umbral")

# 5.6 Proceso de resumen por hora (desnormalización intencional, sección 4.5)
# Es idempotente: si se corre dos veces sobre la misma hora, sobrescribe (INSERT OR REPLACE)
cur5.execute('''
INSERT OR REPLACE INTO lectura_resumen_hora (id_sensor, hora, promedio, minimo, maximo)
SELECT id_sensor,
       strftime('%Y-%m-%d %H:00:00', momento) AS hora,
       AVG(valor), MIN(valor), MAX(valor)
FROM lectura
GROUP BY id_sensor, hora
''')
conn5.commit()
print("Resumen por hora generado:")
display(pd.read_sql_query("SELECT * FROM lectura_resumen_hora", conn5))

# 5.7 Consultas Q-01 a Q-08 del expediente (sección 4.7)

print("Q-04 · Alertas pendientes de la máquina 1:")
display(pd.read_sql_query(
    "SELECT * FROM vista_alertas_pendientes WHERE id_maquina = 1", conn5))

print("Q-05 · Promedio por sensor (tabla lectura_resumen_hora):")
display(pd.read_sql_query(
    "SELECT id_sensor, AVG(promedio) AS promedio_general FROM lectura_resumen_hora GROUP BY id_sensor", conn5))

print("Q-06 · Sensor con más alertas:")
display(pd.read_sql_query(
    "SELECT id_sensor, COUNT(*) AS n_alertas FROM alerta GROUP BY id_sensor ORDER BY n_alertas DESC", conn5))

print("Q-08 · Última lectura de cada sensor (vista_sensores_sin_reportar):")
display(pd.read_sql_query("SELECT * FROM vista_sensores_sin_reportar", conn5))

# 5.8 Rendimiento: EXPLAIN QUERY PLAN sobre Q-01 (inserción) y una consulta por rango de tiempo
plan = cur5.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM lectura WHERE id_sensor = 1 AND momento BETWEEN '2026-09-10 08:00:00' AND '2026-09-10 08:05:00'"
).fetchall()
for fila in plan:
    print(fila)

# 5.9 Prueba de seguridad negativa (RF/seguridad 4.9): el rol "sensor" solo puede insertar lecturas
# En SQLite no hay roles nativos como en Postgres; aquí se simula la restricción a nivel de aplicación,
# comprobando que la función del servicio de ingesta no expone operaciones sobre 'alerta'.
class ServicioIngesta:
    """Simula el rol 'sensor': solo puede insertar lecturas."""
    def __init__(self, cur):
        self._cur = cur
    def insertar_lectura(self, id_sensor, momento, valor):
        self._cur.execute("INSERT INTO lectura VALUES (?,?,?)", (id_sensor, momento, valor))
    # Deliberadamente NO se expone ningún método para leer o modificar 'alerta'

ingesta = ServicioIngesta(cur5)
print("OK — el objeto ServicioIngesta no tiene métodos para leer/modificar alertas:")
print([m for m in dir(ingesta) if not m.startswith('_')])

# ======================================================================
# ## Caso 5 · Ampliación: JOINs y consultas avanzadas sobre series de tiempo
#
# Esta sección amplía el prototipo con:
#
# - **Datos adicionales**: un sensor que nunca reporta, una máquina sin sensores y dos *huecos* de comunicación simulados.
# - **JOINs**: 3 tablas, `LEFT JOIN` en cascada, *anti-join* de consistencia y *self join* entre sensores de la misma máquina.
# - **Consultas avanzadas**: media móvil, `LAG`, top-N por grupo, percentiles, detección de huecos, agrupación de alertas en incidentes (*gaps and islands*) y una vista de estado actual.
# ======================================================================

# 5.10 Datos adicionales (idempotente) + simulación de caídas de comunicación
cur5.execute("INSERT OR IGNORE INTO maquina VALUES (3, 'Horno D', 'Producción')")      # sin sensores
cur5.execute("INSERT OR IGNORE INTO sensor  VALUES (4, 2, 'caudal', 5.0, 50.0)")        # sensor que nunca reporta

# Dos "huecos" de comunicación: se eliminan lecturas para que luego se detecten con LAG
cur5.execute("""DELETE FROM lectura WHERE id_sensor = 2
                AND momento BETWEEN '2026-09-10 08:05:00' AND '2026-09-10 08:05:29'""")
cur5.execute("""DELETE FROM lectura WHERE id_sensor = 3
                AND momento BETWEEN '2026-09-10 08:10:00' AND '2026-09-10 08:10:44'""")
conn5.commit()

# El resumen por hora es idempotente (INSERT OR REPLACE): se recalcula con los datos actuales
cur5.execute('''
INSERT OR REPLACE INTO lectura_resumen_hora (id_sensor, hora, promedio, minimo, maximo)
SELECT id_sensor, strftime('%Y-%m-%d %H:00:00', momento) AS hora, AVG(valor), MIN(valor), MAX(valor)
FROM lectura GROUP BY id_sensor, hora
''')
conn5.commit()

# Q-08b · Sensores realmente inactivos: sin lecturas, o cuya última lectura está > 60 s atrás del
# dato más reciente de la planta. Se filtra sobre la vista original (el DDL del expediente no cambia).
print("Q-08b · Sensores sin reportar (sin lecturas o > 60 s de retraso):")
display(pd.read_sql_query("""
    SELECT * FROM vista_sensores_sin_reportar
    WHERE ultima_lectura IS NULL
       OR ultima_lectura < (SELECT datetime(MAX(momento), '-60 seconds') FROM lectura)
""", conn5))

for tabla in ("maquina", "sensor", "lectura", "alerta", "lectura_resumen_hora"):
    n = cur5.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"{tabla}: {n} filas")

def consulta(conn, titulo, sql):
    print(titulo)
    df = pd.read_sql_query(sql, conn)
    display(df)
    return df

# ======================================================================
# ### JOINs
# ======================================================================

# J-01 · INNER JOIN de 3 tablas: alertas con su máquina, área, tipo de sensor y cuánto excedieron el límite
_ = consulta(conn5, "J-01 · Alertas con contexto (alerta + sensor + máquina), mayores excesos primero:", '''
SELECT a.id_alerta, ma.nombre AS maquina, ma.area, s.tipo AS sensor,
       a.momento, a.valor,
       CASE WHEN a.valor > s.limite_max THEN s.limite_max ELSE s.limite_min END AS limite_violado,
       ROUND(CASE WHEN a.valor > s.limite_max THEN a.valor - s.limite_max
                  ELSE s.limite_min - a.valor END, 2) AS exceso,
       a.estado
FROM alerta a
JOIN sensor  s  ON s.id_sensor  = a.id_sensor
JOIN maquina ma ON ma.id_maquina = s.id_maquina
ORDER BY exceso DESC
LIMIT 10
''')

# J-02 · LEFT JOIN en cascada: inventario máquina -> sensor -> lecturas (muestra lo que no tiene datos)
_ = consulta(conn5, "J-02 · Inventario máquina -> sensor -> lecturas (LEFT JOIN en cascada):", '''
SELECT ma.nombre AS maquina, s.id_sensor, s.tipo,
       COUNT(l.momento) AS n_lecturas,
       MIN(l.momento)   AS primera,
       MAX(l.momento)   AS ultima
FROM maquina ma
LEFT JOIN sensor  s ON s.id_maquina = ma.id_maquina
LEFT JOIN lectura l ON l.id_sensor  = s.id_sensor
GROUP BY ma.id_maquina, s.id_sensor
ORDER BY ma.id_maquina, s.id_sensor
''')

# J-03 · ANTI-JOIN de consistencia (RN-01): toda lectura fuera de rango debe tener su alerta
_ = consulta(conn5, "J-03 · Excepción: lecturas fuera de rango SIN alerta (debe dar 0):", '''
SELECT COUNT(*) AS lecturas_sin_alerta
FROM lectura l
JOIN sensor s ON s.id_sensor = l.id_sensor
LEFT JOIN alerta a ON a.id_sensor = l.id_sensor AND a.momento = l.momento
WHERE (l.valor < s.limite_min OR l.valor > s.limite_max)
  AND a.id_alerta IS NULL
''')

# J-04 · SELF JOIN sobre lectura: temperatura vs vibración de la misma máquina en el mismo instante
df_corr = consulta(conn5, "J-04 · Pares (temperatura, vibración) de la Extrusora A (muestra):", '''
SELECT l1.momento, l1.valor AS temperatura, l2.valor AS vibracion
FROM lectura l1
JOIN sensor s1 ON s1.id_sensor = l1.id_sensor AND s1.tipo = 'temperatura'
JOIN sensor s2 ON s2.id_maquina = s1.id_maquina AND s2.tipo = 'vibracion'
JOIN lectura l2 ON l2.id_sensor = s2.id_sensor AND l2.momento = l1.momento
ORDER BY l1.momento
LIMIT 10
''')
pares = pd.read_sql_query('''
SELECT l1.valor AS temperatura, l2.valor AS vibracion
FROM lectura l1
JOIN sensor s1 ON s1.id_sensor = l1.id_sensor AND s1.tipo = 'temperatura'
JOIN sensor s2 ON s2.id_maquina = s1.id_maquina AND s2.tipo = 'vibracion'
JOIN lectura l2 ON l2.id_sensor = s2.id_sensor AND l2.momento = l1.momento
''', conn5)
print(f"Correlación temperatura-vibración sobre {len(pares)} pares: {pares['temperatura'].corr(pares['vibracion']):.3f}")
print("(Datos simulados de forma independiente, por eso se espera una correlación cercana a 0)")

# J-05 · JOIN entre la tabla resumen y sensor: horas cuyo máximo superó el límite
_ = consulta(conn5, "J-05 · Horas con picos fuera de límite (resumen + sensor + máquina):", '''
SELECT ma.nombre AS maquina, s.tipo, r.hora,
       ROUND(r.promedio, 2) AS promedio, r.minimo, r.maximo, s.limite_min, s.limite_max
FROM lectura_resumen_hora r
JOIN sensor s   ON s.id_sensor   = r.id_sensor
JOIN maquina ma ON ma.id_maquina = s.id_maquina
WHERE r.maximo > s.limite_max OR r.minimo < s.limite_min
ORDER BY r.hora, ma.nombre
''')

# ======================================================================
# ### Consultas avanzadas
# ======================================================================

# A-01 · Media móvil con ventana deslizante (5 lecturas) -> suaviza el ruido del sensor
_ = consulta(conn5, "A-01 · Temperatura vs media móvil de 5 lecturas (sensor 1, primeros 12 s):", '''
SELECT id_sensor, momento, valor,
       ROUND(AVG(valor) OVER (PARTITION BY id_sensor ORDER BY momento
                              ROWS BETWEEN 4 PRECEDING AND CURRENT ROW), 2) AS media_movil_5
FROM lectura
WHERE id_sensor = 1 AND momento < '2026-09-10 08:00:12'
ORDER BY momento
''')

# A-02 · LAG + ROW_NUMBER: las 3 variaciones más bruscas entre lecturas consecutivas por sensor
_ = consulta(conn5, "A-02 · Top 3 saltos bruscos por sensor (LAG + ROW_NUMBER):", '''
WITH delta AS (
  SELECT id_sensor, momento, valor,
         valor - LAG(valor) OVER (PARTITION BY id_sensor ORDER BY momento) AS variacion
  FROM lectura
),
ranking AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY id_sensor ORDER BY ABS(variacion) DESC) AS rn
  FROM delta
  WHERE variacion IS NOT NULL
)
SELECT id_sensor, momento, valor, ROUND(variacion, 2) AS variacion
FROM ranking
WHERE rn <= 3
ORDER BY id_sensor, rn
''')

# A-03 · Detección de huecos (sensores que dejaron de reportar): diferencia > 1 s entre lecturas
_ = consulta(conn5, "A-03 · Huecos de comunicación (> 1 s sin lecturas):", '''
WITH diffs AS (
  SELECT id_sensor, momento,
         LAG(momento) OVER (PARTITION BY id_sensor ORDER BY momento) AS anterior
  FROM lectura
)
SELECT d.id_sensor, ma.nombre AS maquina, d.anterior AS ultima_antes_del_hueco,
       d.momento AS primera_despues,
       CAST(strftime('%s', d.momento) AS INTEGER) - CAST(strftime('%s', d.anterior) AS INTEGER) AS segundos_sin_datos
FROM diffs d
JOIN sensor s   ON s.id_sensor   = d.id_sensor
JOIN maquina ma ON ma.id_maquina = s.id_maquina
WHERE d.anterior IS NOT NULL
  AND CAST(strftime('%s', d.momento) AS INTEGER) - CAST(strftime('%s', d.anterior) AS INTEGER) > 1
ORDER BY d.id_sensor, d.momento
''')

# A-04 · Percentiles (SQLite no trae PERCENTILE): método "nearest-rank" con ROW_NUMBER
_ = consulta(conn5, "A-04 · Percentiles p50 / p95 / p99 por sensor:", '''
WITH ord AS (
  SELECT id_sensor, valor,
         ROW_NUMBER() OVER (PARTITION BY id_sensor ORDER BY valor) AS rn,
         COUNT(*)     OVER (PARTITION BY id_sensor)                AS n
  FROM lectura
)
SELECT o.id_sensor, s.tipo, MIN(o.valor) AS minimo,
       MIN(CASE WHEN o.rn >= 0.50 * o.n THEN o.valor END) AS p50,
       MIN(CASE WHEN o.rn >= 0.95 * o.n THEN o.valor END) AS p95,
       MIN(CASE WHEN o.rn >= 0.99 * o.n THEN o.valor END) AS p99,
       MAX(o.valor) AS maximo
FROM ord o JOIN sensor s ON s.id_sensor = o.id_sensor
GROUP BY o.id_sensor
ORDER BY o.id_sensor
''')

# A-05 · Agregación por minuto + porcentaje de lecturas fuera de rango (JOIN con límites)
_ = consulta(conn5, "A-05 · Minutos con lecturas fuera de rango (% sobre lecturas del minuto):", '''
SELECT s.id_sensor, s.tipo, strftime('%H:%M', l.momento) AS minuto,
       COUNT(*) AS lecturas,
       SUM(l.valor < s.limite_min OR l.valor > s.limite_max) AS fuera_de_rango,
       ROUND(100.0 * SUM(l.valor < s.limite_min OR l.valor > s.limite_max) / COUNT(*), 2) AS pct_fuera
FROM lectura l
JOIN sensor s ON s.id_sensor = l.id_sensor
GROUP BY s.id_sensor, minuto
HAVING fuera_de_rango > 0
ORDER BY pct_fuera DESC, s.id_sensor, minuto
LIMIT 10
''')

# A-06 · Gaps and islands: agrupar alertas del mismo sensor separadas por <= 5 min en un único "incidente"
_ = consulta(conn5, "A-06 · Incidentes (alertas del mismo sensor con menos de 5 min entre sí):", '''
WITH marcadas AS (
  SELECT id_sensor, momento, valor,
         CASE WHEN CAST(strftime('%s', momento) AS INTEGER)
                 - CAST(strftime('%s', LAG(momento) OVER (PARTITION BY id_sensor ORDER BY momento)) AS INTEGER) <= 300
              THEN 0 ELSE 1 END AS inicia_incidente
  FROM alerta
),
agrupadas AS (
  SELECT *, SUM(inicia_incidente) OVER (PARTITION BY id_sensor ORDER BY momento) AS incidente
  FROM marcadas
)
SELECT id_sensor, incidente, MIN(momento) AS inicio, MAX(momento) AS fin,
       COUNT(*) AS n_alertas, ROUND(MAX(valor), 2) AS valor_pico
FROM agrupadas
GROUP BY id_sensor, incidente
ORDER BY id_sensor, incidente
''')

# A-07 · CTEs + LEFT JOIN: tasa de alertas por cada 1.000 lecturas, por máquina
_ = consulta(conn5, "A-07 · Alertas por cada 1.000 lecturas, por máquina:", '''
WITH lect AS (
  SELECT s.id_maquina, COUNT(*) AS n_lecturas
  FROM lectura l JOIN sensor s ON s.id_sensor = l.id_sensor
  GROUP BY s.id_maquina
),
ale AS (
  SELECT s.id_maquina, COUNT(*) AS n_alertas
  FROM alerta a JOIN sensor s ON s.id_sensor = a.id_sensor
  GROUP BY s.id_maquina
)
SELECT ma.nombre AS maquina, ma.area,
       COALESCE(lect.n_lecturas, 0) AS lecturas,
       COALESCE(ale.n_alertas, 0)   AS alertas,
       ROUND(1000.0 * COALESCE(ale.n_alertas, 0) / NULLIF(lect.n_lecturas, 0), 2) AS alertas_por_1000
FROM maquina ma
LEFT JOIN lect ON lect.id_maquina = ma.id_maquina
LEFT JOIN ale  ON ale.id_maquina  = ma.id_maquina
ORDER BY alertas_por_1000 DESC
''')

# A-08 · Estado actual de cada sensor: última lectura (ROW_NUMBER) comparada con sus límites
_ = consulta(conn5, "A-08 · Estado actual por sensor (última lectura vs límites):", '''
WITH ultima AS (
  SELECT id_sensor, momento, valor,
         ROW_NUMBER() OVER (PARTITION BY id_sensor ORDER BY momento DESC) AS rn
  FROM lectura
)
SELECT s.id_sensor, ma.nombre AS maquina, s.tipo,
       u.momento AS ultima_lectura, u.valor,
       CASE WHEN u.momento IS NULL                        THEN 'SIN DATOS'
            WHEN u.valor < s.limite_min OR u.valor > s.limite_max THEN 'FUERA DE RANGO'
            ELSE 'NORMAL' END AS estado
FROM sensor s
JOIN maquina ma ON ma.id_maquina = s.id_maquina
LEFT JOIN ultima u ON u.id_sensor = s.id_sensor AND u.rn = 1
ORDER BY s.id_sensor
''')

# A-09 · Alertas pendientes vs atendidas por máquina (pivot con SUM(CASE))
_ = consulta(conn5, "A-09 · Alertas por máquina y estado (pivot):", '''
SELECT ma.nombre AS maquina,
       COALESCE(SUM(a.estado = 'pendiente'), 0) AS pendientes,
       COALESCE(SUM(a.estado = 'atendida'), 0)  AS atendidas,
       COUNT(a.id_alerta)                       AS total
FROM maquina ma
LEFT JOIN sensor s ON s.id_maquina = ma.id_maquina
LEFT JOIN alerta a ON a.id_sensor  = s.id_sensor
GROUP BY ma.id_maquina
ORDER BY total DESC
''')

# 5.11 Vista nueva basada en JOINs: salud por sensor (cantidad de lecturas, alertas y última lectura)
cur5.executescript('''
DROP VIEW IF EXISTS vista_salud_sensor;
CREATE VIEW vista_salud_sensor AS
  SELECT s.id_sensor, ma.nombre AS maquina, ma.area, s.tipo,
         (SELECT COUNT(*) FROM lectura l WHERE l.id_sensor = s.id_sensor) AS n_lecturas,
         (SELECT MAX(l.momento) FROM lectura l WHERE l.id_sensor = s.id_sensor) AS ultima_lectura,
         (SELECT COUNT(*) FROM alerta a WHERE a.id_sensor = s.id_sensor) AS n_alertas,
         (SELECT COUNT(*) FROM alerta a WHERE a.id_sensor = s.id_sensor AND a.estado = 'pendiente') AS alertas_pendientes
  FROM sensor s JOIN maquina ma ON ma.id_maquina = s.id_maquina;
''')
conn5.commit()
_ = consulta(conn5, "Vista nueva · vista_salud_sensor:", "SELECT * FROM vista_salud_sensor")

# 5.12 Rendimiento: el JOIN con filtro por sensor y rango de tiempo debe usar la PK (id_sensor, momento)
print("EXPLAIN QUERY PLAN · JOIN lectura + sensor con rango de tiempo:")
for fila in cur5.execute('''
    EXPLAIN QUERY PLAN
    SELECT s.tipo, l.momento, l.valor
    FROM lectura l JOIN sensor s ON s.id_sensor = l.id_sensor
    WHERE l.id_sensor = 1 AND l.momento BETWEEN '2026-09-10 08:00:00' AND '2026-09-10 08:05:00'
''').fetchall():
    print(fila)

# ======================================================================
# ## Resumen de lo verificado en este notebook (Caso 5)
#
# | Prueba | Resultado |
# |---|---|
# | Integridad (PK id_sensor + momento) | ✅ RN-02 |
# | Regla de negocio en aplicación | ✅ RN-03 (alerta atendida no se reabre) |
# | Consultas Q-01…Q-08 (+ Q-08b) | ✅ ejecutadas |
# | Rendimiento (EXPLAIN QUERY PLAN) | ✅ |
# | Seguridad (rol restringido simulado) | ✅ ServicioIngesta |
# | JOINs (3 tablas, LEFT en cascada, anti-join, self join) | ✅ J-01 a J-05 |
# | Consultas avanzadas (ventanas, percentiles, huecos, incidentes) | ✅ A-01 a A-09 |
# | Vista nueva basada en JOINs | ✅ vista_salud_sensor |
#
# El archivo `caso5_iot.db` queda en el entorno de Colab y puede descargarse desde el panel de archivos.
# ======================================================================

conn5.close()
